# 05 · CDSE Process API: VV and VH radar

**Objective:** Retrieve linear terrain-normalized radar backscatter on one orbit direction and inspect VV, VH, and their ratio.

**Route:** Live API / CDSE OAuth client · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** radar maps, summary statistics and explicit processing metadata. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/cdse-sentinel-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Configure comparable radar processing

Use IW dual-polarization acquisitions, fixed ascending orbit direction, terrain-normalized gamma0 and orthorectification with Copernicus 90 m DEM. The 30 m DEM option can require extra CDSE registration. VV and VH remain **linear power**. The ratio is VV/VH in linear units; in decibels the corresponding contrast is VV_dB − VH_dB.


In [ ]:
from examples.sentinel import access_token, process_payload, process
bbox = [24.55, 60.15, 24.57, 60.17]
start, end = "2024-08-01T00:00:00Z", "2024-08-31T23:59:59Z"
orbit = "ASCENDING"
payload = process_payload("sentinel-1-grd", bbox, start, end, orbit=orbit)
print(payload["input"])


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
token = access_token()
image, grid = process(token, payload)
vv, vh, mask = image
valid = (mask > 0) & np.isfinite(vv) & np.isfinite(vh) & (vv > 0) & (vh > 0)
ratio = np.full(vv.shape, np.nan)
np.divide(vv, vh, out=ratio, where=valid)
vv_db = np.where(valid, 10 * np.log10(np.maximum(vv, 1e-12)), np.nan)
vh_db = np.where(valid, 10 * np.log10(np.maximum(vh, 1e-12)), np.nan)
assert valid.any()
pd.DataFrame({"VV_linear": vv[valid], "VH_linear": vh[valid], "VV_VH_linear_ratio": ratio[valid]}).describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, arr, title in zip(axes, [vv_db, vh_db, ratio], ["VV (dB)", "VH (dB)", "VV/VH (linear)"]):
    im = ax.imshow(arr)
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.show()
record_provenance("sentinel-1-process.json", collection="sentinel-1-grd", request=payload, grid=grid, orbit_direction=orbit, coefficient="GAMMA0_TERRAIN", valid_pixels=int(valid.sum()))


## Interpret and verify

Radar depends on incidence angle, roughness, vegetation, moisture, orbit and speckle. A VV/VH change is not a calibrated water-table measurement. Whole-month mosaicking can mix dates; use the next radar time-series tutorial for narrower intervals.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [CDSE Process API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Process.html)
- [CDSE authentication](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Overview/Authentication.html)
- [Sentinel-1 processing options](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S1GRD.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
